# Main Code for NYTC Qualification (1,2,3 combined)
Made by ***N2IC Team***

## Setup

In [1]:
import importlib
import time
import cv2
import numpy as np
from ugot import ugot
import pose_yolo
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.


# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address.
# Change this string if your robot is on a different IP.
got.initialize("192.168.88.1")

# Load the built-in computer vision models that will be used later in the notebook.
# You can load additional models here if needed — just add the model name to the list.
got.load_models(
    [
        "color_recognition",  # detects dominant colors 
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode" # AprilTag recognition
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("Done!")

ModuleNotFoundError: No module named 'pose_yolo'

## Combined Code

### Function Init

In [ ]:
def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drive toward a detected AprilTag, keeping it centered in the camera frame.

    Parameters:
        distance  (float): Stop when the tag is within this many meters (default 0.15 m).
        gap       (int):   Pixel tolerance around center (320 px) before strafing (default 20 px).
        fwd_spd   (int):   Forward drive speed percentage (default 10 cm/s).
        strafe_spd(int):   Left/right correction speed percentage (default 10 cm/s).
    """
    # Get an initial reading to confirm a tag is visible before entering the loop.
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1] # Horizontal pixel position of the tag (0=left, 640=right)
    AP_distance = AP_info[0][6] # Estimated distance to the tag in meters

    while True:
        try:
            # Refresh tag data every iteration for responsive corrections.
            AP_info = got.get_apriltag_total_info()
            AP_x = AP_info[0][1]
            AP_distance = AP_info[0][6]

            if AP_x < 320 - gap:
                # Tag is to the LEFT of center — strafe left to re-align.
                # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
                got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
            elif AP_x > 320 + gap:
                # Tag is to the RIGHT of center — strafe right to re-align.
                got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
            elif AP_distance > distance:
                # Tag is centered but still too far — drive straight forward.
                got.mecanum_move_xyz(0, fwd_spd, 0)
            else:
                # Tag is centered AND within target distance — stop and exit.
                got.mecanum_stop()
                print("It's too close, let's stop.")
                break
        except IndexError:
            print("AprilTag is too close!")


def pick_up():
    """
    Pick up the object identified by the closest visible AprilTag.

    Assumes the robot is already aligned and close to the target
    (i.e., AP_centralization_approaching() has just completed).
    """
    # Read the tag's current position and distance for arm targeting.
    AP_info = got.get_apriltag_total_info()
    try:
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # Move arm to a neutral ready position and open the gripper.
        # joint_control(j1, j2, j3, duration_ms): j2=30, j3=30 tilts arm slightly forward.
        got.mechanical_joint_control(0, 30, 30, 1000)
        got.mechanical_clamp_release() # Open gripper before extending arm
        time.sleep(2) # Wait for gripper to fully open

        # Calculate arm joint angles based on the tag's camera position.
        # joint1 (base): convert pixel offset from center to degrees.
        #   Negative factor corrects for the camera being mirrored horizontally.
        joint1 = int((AP_x - 320) * -1 / 10)

        # joint3 (furthest): convert distance (m) to an extension angle.
        # The -70 offset accounts for the arm's resting angle calibration.
        joint3 = int(AP_distance * 100 - 70)

        # Move arm to the computed pick-up pose.
        got.mechanical_joint_control(joint1, 0, joint3, 500)
        print(f"Joint1 value is: {joint1}, Joint3 value is: {joint3}.")
        time.sleep(1) # Wait for arm to reach the target pose

        # Grasp the object and lift the arm back to the carry position.
        got.mechanical_clamp_close()
        time.sleep(2)  # Wait for gripper to fully close before lifting
        got.mechanical_joint_control(0, 30, 30, 1000)  # Return arm to neutral carry pose
    except IndexError:
        print("AprilTag is too close!")



# Line Tracing:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, line_type, x, y = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type, x, y


def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then approach them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # Phase 1: Spin and search
        while True:
            # Turn slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a name tag)
            name = got.get_words_result()

            # Check for any recognized faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective turn to center the robot on the target
                # turn=3 is clockwise; times=10, unit=2 means ~10 degree-units
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # Phase 2: Approach the target
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face; inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
                h = faces[0][3]  # Height of the face bounding box (proxy for distance)
                if h < height:
                    if c_x < 320 - gap:
                        # Face is too far LEFT — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is too far RIGHT — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centered but still small (far away) — move straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break  # Done

            clear_output(wait=True)

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

### Run

In [ ]:
TARGET_NAME = "Ryan" # Key in the name of the target person to find in face_find_and_approach()

In [ ]:
# Challenge 1: Color Detection and Conditional Action
print("Starting challenge 1: Look for red, then green, then pick up the AprilTag...")
# Try to look for red and green.
print("Looking for red and green...")
red_seen = False
try:
    while True:
        color = got.get_color_total_info()[0]
        if color == "Green" and red_seen:
            got.screen_display_background(6)
            print("Green detected")
            time.sleep(0.5)
            got.screen_display_background(0)
            break
        elif color == "Red":
            print("Red seen")
            got.screen_display_background(3)
            red_seen = True

    # If see green, locate and pickup AprilTag
    print("Looking for AprilTag...")
    while True:
        AP_info = got.get_apriltag_total_info()

        if AP_info:
            print("see tag")
            AP_centralization_approaching(
                distance=0.14, gap=18, fwd_spd=11, strafe_spd=5
            )
            pick_up()
            print("Picked up!")
            time.sleep(2)
            break
        else:
            print("see no tag, moving up")
            got.mecanum_move_speed(0, 15)

    # Challenge 2: Line Following with Conditional Turns
    print("Starting challenge 2: Follow the line, then turn based on camera text input")
    #After picking up, move forward with line tracing until sees a T intersection, then stop.
    print("Following line until T intersection...")
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=15)
        if line_type in [0,2]:
            print("See the intersection, stopping.")
            got.mecanum_stop()
            break
    # see, intersection, move forward a bit to fully cross the intersection, then stop.
    got.mecanum_translate_speed_times(angle=0, speed=20, times=13, unit=1)
    

    # Read left or right and turn accordingly until see the line again, then stop.
    print("Looking for LEFT or RIGHT command...")
    while True:
        # Read any text currently visible in the camera frame.
        text = got.get_words_result()

        print("Text:", text)

        # React to specific command words
            # Turn counter-clockwise by ~45 degrees
        if text in ["LEFT", "RIGHT"]:
            turn = 2
            if(text == "RIGHT"): turn = 3
            while True:
                got.mecanum_turn_speed_times(turn=turn, speed=30, times=20, unit=2)
                line_type, _, _ = line_follow(mult=0.25, speed=0)
                if line_type == 1:
                    print("Turned and see line")
                    break
            break

    # Go the rest of the way to the end of the line and stop.
    print("Going to the end of the line...")
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=15)
        if line_type in [0]:
            print("can not see path anymore, stopping.")
            break 
    
    #Challenge 3: Face Recognition and Personalized Greeting
    print("Starting challenge 3: Look for a registered face and greet them by name")
    #Pose recognition control: look for registered faces, and if see one, greet them by name on the screen.
    print("Starting pose recognition...")
    run_pose_control_inline(
        robot_ip="192.168.88.1",
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=1,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
        got=got,
    )

    # Looking for face and move towards box, release april tag
    print("Starting face recognition...")
    face_find_and_approach(
        gap=15,
        target_name=TARGET_NAME,
        turn_spd=10,
        strafe_spd=10,
        fwd_spd=10,
        height=120,
        adjust_turn=10,
    )
    got.mecanum_move_speed_times(
        direction=1, speed=20, times=15, unit=1
    )  # Go backwards a bit to not hit face
    got.mechanical_joint_control(angle1=0, angle2=-30, angle3=-30, duration=500)
    time.sleep(1)
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)
    got.mechanical_clamp_release()

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    print("Done")

Red seen
Red seen
Red seen
Red seen
Red seen
Red seen
Red seen
Red seen
Red seen
Red seen
Red seen
Green detected
Starting line follow...
